# Hasaki Crawler

Crawl products + Q&A từ 3 categories:
- Sức khỏe làm đẹp (c3)
- Mỹ phẩm high end (c1907)
- Trang điểm (c23)

In [6]:
# !pip install playwright
# !playwright install chromium

In [7]:
import json
import time
import re
import os
from datetime import datetime
from collections import Counter
from multiprocessing import Process
from playwright.async_api import async_playwright

## Config

In [8]:
BASE_URL = "https://hasaki.vn"
CATEGORIES = [
    ("suc-khoe-lam-dep-c3", "Sức khỏe làm đẹp"),
    ("my-pham-high-end-c1907", "Mỹ phẩm high end"),
    ("trang-diem-c23", "Trang điểm"),
]
PAGES_PER_CATEGORY = 10
NUM_QA_BATCHES = 4

USER_AGENT = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

## Part 1: Crawl Products

In [9]:
def extract_product_id(url):
    match = re.search(r'/san-pham/([^/?#]+?)(?:\.html)?(?:\?|$|#)', url)
    return match.group(1) if match else url.split('/')[-1].replace('.html', '')


async def crawl_category(page, category_path, category_name, num_pages=10):
    products = []
    seen_urls = set()

    for pg in range(1, num_pages + 1):
        url = f"{BASE_URL}/danh-muc/{category_path}.html"
        if pg > 1:
            url += f"?page={pg}"

        print(f"  Page {pg}/{num_pages}", end="")
        try:
            await page.goto(url, wait_until="networkidle", timeout=30000)
        except Exception:
            await page.goto(url, wait_until="domcontentloaded", timeout=30000)
        time.sleep(3)

        for _ in range(3):
            await page.evaluate("window.scrollBy(0, 800)")
            time.sleep(0.5)

        links = await page.evaluate("""
            () => {
                const seen = new Set(), links = [];
                document.querySelectorAll('a[href*="/san-pham/"]').forEach(a => {
                    if (a.href && !seen.has(a.href)) {
                        seen.add(a.href);
                        const name = a.textContent.trim().substring(0, 300) || '';
                        const img = a.querySelector('img');
                        const imgSrc = img ? (img.src || img.dataset.src || '') : '';
                        links.push({href: a.href, name, imgSrc});
                    }
                });
                return links;
            }
        """)

        if not links:
            print(" -> end of category")
            break

        new_count = 0
        for link in links:
            if link['href'] not in seen_urls:
                seen_urls.add(link['href'])
                new_count += 1
                products.append({
                    'product_id': extract_product_id(link['href']),
                    'product_name_raw': link['name'],
                    'product_url': link['href'],
                    'category_name': category_name,
                    'image_url': link['imgSrc'],
                })

        print(f" -> {new_count} new (total: {len(products)})")
        if new_count == 0:
            break
        time.sleep(1.5)

    return products

In [10]:
all_products = []

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    ctx = await browser.new_context(user_agent=USER_AGENT, locale="vi-VN")
    page = await ctx.new_page()

    for cat_path, cat_name in CATEGORIES:
        print(f"\n=== {cat_name} ===")
        products = await crawl_category(page, cat_path, cat_name, num_pages=PAGES_PER_CATEGORY)
        all_products.extend(products)

    await browser.close()

print(f"\nTotal: {len(all_products)} products")


=== Sức khỏe làm đẹp ===
  Page 1/10 -> 40 new (total: 40)
  Page 2/10 -> 40 new (total: 80)
  Page 3/10 -> 40 new (total: 120)
  Page 4/10 -> 40 new (total: 160)
  Page 5/10 -> 40 new (total: 200)
  Page 6/10 -> 40 new (total: 240)
  Page 7/10 -> 40 new (total: 280)
  Page 8/10 -> 40 new (total: 320)
  Page 9/10 -> 40 new (total: 360)
  Page 10/10 -> 40 new (total: 400)

=== Mỹ phẩm high end ===
  Page 1/10 -> 40 new (total: 40)
  Page 2/10 -> 27 new (total: 67)
  Page 3/10 -> 0 new (total: 67)

=== Trang điểm ===
  Page 1/10 -> 40 new (total: 40)
  Page 2/10 -> 40 new (total: 80)
  Page 3/10 -> 40 new (total: 120)
  Page 4/10 -> 40 new (total: 160)
  Page 5/10 -> 40 new (total: 200)
  Page 6/10 -> 40 new (total: 240)
  Page 7/10 -> 40 new (total: 280)
  Page 8/10 -> 39 new (total: 319)
  Page 9/10 -> 40 new (total: 359)
  Page 10/10 -> 40 new (total: 399)

Total: 866 products


## Part 2: Clean product names

In [11]:
KNOWN_BRANDS = [
    "L'Oreal Professionnel", "La Roche-Posay", "Some By Mi", "Paula's Choice",
    "The Ordinary", "Dear Klairs", "Hada Labo", "Banila Co", "Holika Holika",
    "Bobbi Brown", "Charlotte Tilbury", "Fenty Beauty", "Rare Beauty",
    "Wet n Wild", "Estee Lauder", "Tom Ford",
    "CeraVe", "Cocoon", "Klairs", "Bioderma", "Maybelline", "DHC", "3CE",
    "Obagi", "MartiDerm", "Martiderm", "Vichy", "Eucerin", "Innisfree",
    "Neutrogena", "Garnier", "Nivea", "Rohto", "Senka", "Skin1004",
    "Pond's", "SVR", "Uriage", "Avene", "Heliocare", "Cetaphil",
    "Carslan", "Aperire", "Marvis", "Romand", "Merzy", "Espoir",
    "Peripera", "Laneige", "Sulwhasoo", "SK-II", "Shiseido", "Clinique",
    "MAC", "NARS", "YSL", "Dior", "Chanel", "Lancome", "Kiehl's",
    "Origins", "ColourPop", "NYX", "Revlon", "Catrice", "Essence",
    "Heimish", "Torriden", "Mediheal", "Anessa", "Biore", "Sunplay",
    "Missha", "Etude", "COSRX", "Mamonde", "Judydoll", "Vaseline",
    "LipIce", "Fixderma", "AHC", "Dove", "TRESemme", "Pantene",
]
# Sort longest first for greedy match
KNOWN_BRANDS.sort(key=len, reverse=True)


def clean_product_name(raw_name):
    name = raw_name.split('\n')[0].strip()
    # Remove discount/price prefix
    name = re.sub(r'^-?\d+\s*%', '', name).strip()
    name = re.sub(r'^\d[\d.,]*\s*₫', '', name).strip()
    name = re.sub(r'^\d[\d.,]*\s*₫', '', name).strip()
    name = re.sub(r'^Tặng:.*?₫', '', name).strip()
    # Remove trailing rating
    name = re.sub(r'\d+\.\d+\(\d+\)\d*$', '', name).strip()
    name = re.sub(r'\d+$', '', name).strip()

    # Detect brand
    brand = ''
    for b in KNOWN_BRANDS:
        if name.startswith(b):
            brand = b
            name = name[len(b):].strip()
            break
    if not brand:
        m = re.match(r'^([A-Z][a-z]+(?:[A-Z][a-z]+)+)', name)
        if m:
            brand = m.group(1)
            name = name[len(brand):].strip()

    # Remove English suffix after ml/g
    name = re.sub(r'(\d+(?:ml|g|L))[A-Z][a-z].*$', r'\1', name)
    return name, brand

In [12]:
# Dedup + clean
seen_ids = set()
products_clean = []

for p in all_products:
    if p['product_id'] in seen_ids:
        continue
    seen_ids.add(p['product_id'])

    name, brand = clean_product_name(p['product_name_raw'])
    p['product_name'] = name
    p['brand'] = brand
    products_clean.append(p)

print(f"Unique products: {len(products_clean)}")

# Stats
cats = Counter(p['category_name'] for p in products_clean)
brands = Counter(p['brand'] for p in products_clean if p['brand'])
print(f"\nBy category:")
for c, n in cats.most_common():
    print(f"  {c}: {n}")
print(f"\nTop 10 brands:")
for b, n in brands.most_common(10):
    print(f"  {b}: {n}")

# Sample
for p in products_clean[:5]:
    print(f"  [{p['brand'] or '?'}] {p['product_name'][:70]}")

Unique products: 822

By category:
  Sức khỏe làm đẹp: 400
  Trang điểm: 364
  Mỹ phẩm high end: 58

Top 10 brands:
  Maybelline: 62
  3CE: 61
  Judydoll: 39
  Cocoon: 30
  Vaseline: 18
  Marvis: 14
  L'Oreal Professionnel: 11
  LipIce: 10
  Skin1004: 9
  CeraVe: 8
  [CeraVe] Sữa Rửa Mặt CeraVe Sạch Sâu Cho Da Thường Đến Da Dầu 473ml
  [Cocoon] Combo 2 Nước Tẩy Trang Bí Đao Cocoon Làm Sạch & Giảm Dầu 500ml
  [Cocoon] Combo Cocoon Nước Cân Bằng Sen Hậu Giang 310ml + Nước Tẩy Trang Bí Đao
  [Klairs] Nước Hoa Hồng Klairs Không Mùi Cho Da Nhạy Cảm 180ml
  [Bioderma] Nước Tẩy Trang Bioderma Dành Cho Da Nhạy Cảm 500ml


In [13]:
# Save products
with open('hasaki_products_full.json', 'w', encoding='utf-8') as f:
    json.dump(products_clean, f, ensure_ascii=False, indent=2)

print(f"Saved {len(products_clean)} products -> hasaki_products_full.json")

Saved 822 products -> hasaki_products_full.json


## Part 3: Crawl Q&A (4 batches song song)

In [18]:
async def extract_questions(page, product):
    """Extract câu hỏi khách hàng từ 1 product page."""
    url = product['product_url']
    pid = product['product_id']
    name = product.get('product_name', '')

    try:
        await page.goto(url, wait_until='domcontentloaded', timeout=12000)
    except Exception:
        return []
    time.sleep(1)

    # Click Q&A tab
    await page.evaluate("""
        () => {
            for (const a of document.querySelectorAll('a')) {
                if (a.textContent.trim().startsWith('Hỏi đáp') && a.className.includes('border-b')) {
                    a.click(); break;
                }
            }
        }
    """)
    time.sleep(1)

    raw = await page.evaluate("""
        () => {
            const qs = [];
            let c = null;
            for (const h of document.querySelectorAll('h2')) {
                if (h.textContent.includes('Hỏi đáp')) {
                    c = h.closest('div[class*="shadow"]') || h.parentElement;
                    break;
                }
            }
            if (!c) return qs;

            for (const block of c.querySelectorAll('div[class*="border-b"]')) {
                const top = block.querySelector(':scope > div.text-sm');
                if (!top) continue;
                const userEl = top.querySelector('p.font-bold');
                const user = userEl ? userEl.textContent.trim() : '';
                let q = '';
                for (const p of top.querySelectorAll(':scope > p')) {
                    if (!p.classList.contains('font-bold') && p.textContent.trim().length > 2) {
                        q = p.textContent.trim(); break;
                    }
                }
                const dateEl = top.querySelector('div.text-xs p');
                const date = dateEl ? dateEl.textContent.trim() : '';
                if (q && user && user !== 'Hasaki') qs.push({user, question: q, date});
            }
            return qs;
        }
    """)

    return [{
        'product_id': pid,
        'product_name': name,
        'user': q['user'],
        'question': q['question'],
        'date': q['date'],
    } for q in raw]

In [19]:
# Load products
with open('hasaki_products_full.json', 'r', encoding='utf-8') as f:
    products_for_qa = json.load(f)

print(f"Total: {len(products_for_qa)} products")
print(f"Running {NUM_QA_BATCHES} concurrent browsers...\n")

# Split batches
batch_size = len(products_for_qa) // NUM_QA_BATCHES + 1
batches = [products_for_qa[i:i + batch_size] for i in range(0, len(products_for_qa), batch_size)]

import asyncio

async def run_batch(batch_id, products):
    """Chạy 1 batch với 1 browser riêng."""
    results = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        ctx = await browser.new_context(user_agent=USER_AGENT, locale='vi-VN')
        page = await ctx.new_page()

        for i, prod in enumerate(products):
            try:
                qs = await extract_questions(page, prod)
                results.extend(qs)
            except Exception:
                pass

            if (i + 1) % 30 == 0 or i == len(products) - 1:
                print(f"  [Batch {batch_id}] {i+1}/{len(products)} -> {len(results)} questions")

            time.sleep(0.3)

        await browser.close()
    print(f"  [Batch {batch_id}] DONE: {len(results)} questions")
    return results

# Run all batches concurrently
batch_results = await asyncio.gather(*[
    run_batch(i, batch) for i, batch in enumerate(batches)
])

all_questions = []
for results in batch_results:
    all_questions.extend(results)

# Save
with open('hasaki_questions.json', 'w', encoding='utf-8') as f:
    json.dump(all_questions, f, ensure_ascii=False, indent=2)

print(f"\nTotal: {len(all_questions)} questions -> hasaki_questions.json")

Total: 822 products
Running 4 concurrent browsers...

  [Batch 1] 30/206 -> 150 questions
  [Batch 0] 30/206 -> 150 questions
  [Batch 3] 30/204 -> 125 questions
  [Batch 2] 30/206 -> 51 questions
  [Batch 1] 60/206 -> 298 questions
  [Batch 0] 60/206 -> 300 questions
  [Batch 3] 60/204 -> 242 questions
  [Batch 2] 60/206 -> 138 questions
  [Batch 1] 90/206 -> 448 questions
  [Batch 3] 90/204 -> 363 questions
  [Batch 0] 90/206 -> 450 questions
  [Batch 2] 90/206 -> 287 questions
  [Batch 1] 120/206 -> 597 questions
  [Batch 0] 120/206 -> 595 questions
  [Batch 2] 120/206 -> 415 questions
  [Batch 3] 120/204 -> 463 questions
  [Batch 1] 150/206 -> 747 questions
  [Batch 0] 150/206 -> 742 questions
  [Batch 2] 150/206 -> 547 questions
  [Batch 3] 150/204 -> 572 questions
  [Batch 0] 180/206 -> 891 questions
  [Batch 1] 180/206 -> 887 questions
  [Batch 2] 180/206 -> 686 questions
  [Batch 3] 180/204 -> 693 questions
  [Batch 1] 206/206 -> 996 questions
  [Batch 3] 204/204 -> 752 questio

## Summary

In [20]:
with open('hasaki_questions.json', 'r', encoding='utf-8') as f:
    questions = json.load(f)

products_with_qa = len(set(q['product_id'] for q in questions))

print(f"Total questions: {len(questions)}")
print(f"Products with Q&A: {products_with_qa}")
if products_with_qa:
    print(f"Avg questions/product: {len(questions)/products_with_qa:.1f}")

print(f"\nSample questions:")
for q in questions[:15]:
    print(f"  [{q['product_name'][:30]}] {q['question'][:80]}")

Total questions: 3568
Products with Q&A: 769
Avg questions/product: 4.6

Sample questions:
  [Sữa Rửa Mặt CeraVe Sạch Sâu Ch] Mình mới nhận đơn hàng 2604085OA8MCMG mà thấy thất vọng vô cùng. Shipper Nguyễn 
  [Sữa Rửa Mặt CeraVe Sạch Sâu Ch] shop ơi gửi đơn sớm cho mìn dc ko
  [Sữa Rửa Mặt CeraVe Sạch Sâu Ch] Vậy là sao nv bên bạn nghĩ mih k bít gì hay sao, mih k đôg ý như vậy. Chặn quà c
  [Sữa Rửa Mặt CeraVe Sạch Sâu Ch] sao bên nv chặn quà khuyến mãi của mình vậy, minh mua 2sp. 1 rửa mặt , 1 kem dưỡ
  [Sữa Rửa Mặt CeraVe Sạch Sâu Ch] Bên mình còn tặng túi laptop k ạ
  [Combo 2 Nước Tẩy Trang Bí Đao ] Mẫu chai có dán tem ko shop
  [Combo 2 Nước Tẩy Trang Bí Đao ] Em mua ngoài store thì có được giá như này ko ạ?
  [Combo 2 Nước Tẩy Trang Bí Đao ] Mình muốn mua sản phẩm này
  [Combo 2 Nước Tẩy Trang Bí Đao ] Giá khuyến mãi hiện tại có áp dụng mua trực tiếp tại cửa hàng không ạ
  [Combo 2 Nước Tẩy Trang Bí Đao ] Da khô xài được không ạ
  [Combo Cocoon Nước Cân Bằng Sen] Giá sale ngày đô

## Part 4: Merge & chuẩn hoá data (pre-labeling)

In [7]:
# Load data
import json
import re
from collections import Counter

with open('hasaki_products_full.json', 'r', encoding='utf-8') as f:
    products = json.load(f)
with open('hasaki_questions.json', 'r', encoding='utf-8') as f:
    questions = json.load(f)

# Build product lookup: product_id -> product metadata
product_map = {p['product_id']: p for p in products}

# Detect Wh-question patterns (Vietnamese)
WH_PATTERN = re.compile(
    r'\b(bao giờ|bao lâu|bao nhiêu|khi nào|ở đâu|tại sao|vì sao|như thế nào|thế nào'
    r'|làm sao|sao lại|gì |cái gì|là gì|loại nào|màu nào|size nào|cỡ nào|mấy giờ'
    r'|mấy ngày|ai |của ai|cho ai|từ đâu|ntn)\b',
    re.IGNORECASE
)

# Group questions by product_id to generate sequential sample_id
from collections import defaultdict

pid_counter = defaultdict(int)

dataset = []
for q in questions:
    pid = q['product_id']
    idx = pid_counter[pid]
    pid_counter[pid] += 1

    sentence = q['question'].strip()
    is_wh = bool(WH_PATTERN.search(sentence))

    # Lookup product metadata
    prod = product_map.get(pid, {})

    # Build image field from product image_url
    image_url = prod.get('image_url', '')
    image = {
        'image_urls': [image_url] if image_url else [],
        'local_paths': []
    } if image_url else None

    sample = {
        'sample_id': f"{pid}_q{idx:02d}",
        'product_id': pid,
        'product_name': q.get('product_name', prod.get('product_name', '')),
        'brand': prod.get('brand', ''),
        'category': prod.get('category_name', ''),
        'sentence': sentence,
        'user': q.get('user', ''),
        'date': q.get('date', ''),
        'image': image,
        'spans': [],
        'isWhQuestion': is_wh,
        'intent': {
            'level_1': '',
            'level_2': '',
            'level_3': []
        },
        'modality': 'text',
        'cross_modal_ref': False,
        'difficulty': None,
        'source': 'hasaki'
    }
    dataset.append(sample)

# Save
OUTPUT_PATH = 'hasaki_prelabel.json'
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

# Stats
has_image = sum(1 for s in dataset if s['image'])
print(f"Total samples: {len(dataset)}")
print(f"With image_url: {has_image} ({has_image/len(dataset)*100:.1f}%)")
print(f"Wh-questions: {sum(1 for s in dataset if s['isWhQuestion'])} ({sum(1 for s in dataset if s['isWhQuestion'])/len(dataset)*100:.1f}%)")
print(f"Unique products: {len(pid_counter)}")
print(f"Categories: {Counter(s['category'] for s in dataset).most_common()}")
print(f"Top brands: {Counter(s['brand'] for s in dataset if s['brand']).most_common(5)}")
print(f"\nSaved -> {OUTPUT_PATH}")

# Sample output
print(f"\n--- Sample ---")
print(json.dumps(dataset[0], ensure_ascii=False, indent=2))

Total samples: 3568
With image_url: 3568 (100.0%)
Wh-questions: 566 (15.9%)
Unique products: 769
Categories: [('Sức khỏe làm đẹp', 1976), ('Trang điểm', 1475), ('Mỹ phẩm high end', 117)]
Top brands: [('Maybelline', 288), ('3CE', 253), ('Judydoll', 178), ('Cocoon', 150), ('Vaseline', 85)]

Saved -> hasaki_prelabel.json

--- Sample ---
{
  "sample_id": "sua-rua-mat-cerave-sach-sau-cho-da-thuong-den-da-dau-473ml-102959_q00",
  "product_id": "sua-rua-mat-cerave-sach-sau-cho-da-thuong-den-da-dau-473ml-102959",
  "product_name": "Sữa Rửa Mặt CeraVe Sạch Sâu Cho Da Thường Đến Da Dầu 473ml",
  "brand": "CeraVe",
  "category": "Sức khỏe làm đẹp",
  "sentence": "Mình mới nhận đơn hàng 2604085OA8MCMG mà thấy thất vọng vô cùng. Shipper Nguyễn Văn Tới (0938.091.852) bên Hasaki có thái độ rất kém: nói chuyện trống không với khách, khách vừa chạy ra nhận hàng chưa thấy hàng đâu mà shipper đã đưa mã QR ra bảo khách chuyển tiền, đợi khách chuyển tiền chưa tới 2 phút mà cứ càm ràm bảo sao không ra cửa h